# MSG-RM plate homogenization: the 8×8 ABDG wall law from a 1-D shell YAML

Every wall laminate of a thin-walled cross-section carries an **RM plate constitutive law**

$$\mathrm{ABDG} \;=\; \begin{bmatrix} A & B & 0\\ B & D & 0\\ 0 & 0 & G \end{bmatrix}\;(8\times8),$$

rows 1–6 the classical membrane/bending ABD (plate strains $[\epsilon_{11},\epsilon_{22},\gamma_{12},
\kappa_{11},\kappa_{22},\kappa_{12}]$), rows 7–8 the transverse shear $[2\gamma_{13},2\gamma_{23}]$.
This tutorial computes it for **every laminate of a 1-D shell SG YAML** with the **MSG/VAM
construction** of Yu, Hodges & Volovoi (*Computers & Structures* 81:439–454, 2003 — the plate twin
of the CMAME 2002 shell paper), implemented in core OpenSG as
`opensg_jax.fe_jax.msg_rm_plate.rm_plate_msg`:

| step | paper | code |
|------|-------|------|
| zeroth-order warping $V_0$ → classical ABD | Eq. (39)–(40) | `V0`, `A6` |
| first-order gradient warping $V_{11},V_{12}$ | Eq. (42)–(45) | `C1bar`, `C2bar` |
| second-order gradient energy $B,C,D$ | Eq. (46)–(47) | `H` (12×12) |
| RM projection: least squares of the residual $U^*$ over $X=G^{-1}$ **and Yu's 24 in-plane relaxed constants** (78 equations, 27 unknowns) | Eq. (55)–(60) | `blocks()`, `lstsq` → `G_msg`, `Ustar_rel` |

`Ustar_rel` is the fraction of the second-order gradient energy the RM functional could **not**
absorb — the distance from asymptotic correctness (0 means an exactly asymptotically-correct RM
model exists for that laminate).

**Input (committed in this repo):** `examples/data/1d_yaml/st15_shell.yaml` — the BAR-URC
station-15 blade cross-section (10 wall laminates, glass triax/UD + foam sandwich walls).
Command-line counterpart: `examples/6_get_plateRM_homo_using_1DSG.py`.

In [1]:
import os, sys
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(ROOT, "opensg_jax")):
    ROOT = os.path.abspath(os.getcwd())          # also runs from the repo root
sys.path.insert(0, ROOT)
np.set_printoptions(precision=4, linewidth=150)

from opensg_jax.fe_jax.msg_mesh import load_yaml
from opensg_jax.fe_jax.msg_rm_plate import rm_plate_msg
from opensg_jax.fe_jax.msg_transverse_shear import plate_8x8, transverse_shear_stiffness

SHELL = os.path.join(ROOT, "examples", "data", "1d_yaml", "st15_shell.yaml")
nodes, elems, mdb, layup_db, elem_to_layup = load_yaml(SHELL)
print("%d wall laminates, %d contour elements, materials: %s"
      % (len(layup_db), len(elems), ", ".join(mdb)))

10 wall laminates, 64 contour elements, materials: Gelcoat, Adhesive, glass_uni, carbon_uni_industry_baseline, glass_biax, glass_triax, medium_density_foam, resin, steel


## Sanity check first: homogeneous isotropic plate

For an isotropic single layer with $\nu=0$ the construction must return the textbook
$G = \tfrac{5}{6}\,G\,h$ **exactly**, with $U^*$ driven to machine zero (an asymptotically
correct RM model exists for this case).

In [2]:
h = 0.01
mdb_iso = {"iso": {"E": [70e9]*3, "G": [35e9]*3, "nu": [0.0]*3, "rho": 1.0}}
r = rm_plate_msg([h], [0.0], ["iso"], mdb_iso, fraction=0.5)
print("G_msg/(G*h) =", np.diag(r["G_msg"]) / (35e9*h), "  target 5/6 =", 5/6)
print("Ustar_rel   = %.2e" % r["Ustar_rel"])

G_msg/(G*h) = [0.8333 0.8333]   target 5/6 = 0.8333333333333334
Ustar_rel   = 7.81e-16


## The 8×8 ABDG for one wall laminate

Full matrix for the first laminate at the **center (mid-surface) reference** — the convention
of the RM ring homogenization.  The reference plane is set by `fraction` (0 = OML face,
0.5 = center, 1 = IML face; default 0 = OML).

In [3]:
ln = "layup_0"; lay = layup_db[ln]
thk = [float(t) for t in lay["thick"]]; ang = [float(a) for a in lay["angles"]]
mats = [str(m) for m in lay["mat_names"]]
h = float(sum(thk))
print(ln, ":", ", ".join("%s(%.1fmm/%g)" % (m, 1e3*t, a) for m, t, a in zip(mats, thk, ang)))

r = rm_plate_msg(thk, ang, mats, mdb, fraction=0.5)
P8 = plate_8x8(r["A6"], r["G_msg"])
print("\nRM 8x8 ABDG  [[A,B,0],[B,D,0],[0,0,G]]:")
print(P8)

layup_0 : glass_triax(4.0mm/0), glass_uni(10.0mm/0), glass_triax(2.0mm/0)

RM 8x8 ABDG  [[A,B,0],[B,D,0],[0,0,G]]:
[[ 6.4602e+08  1.0091e+08  0.0000e+00  1.1964e+05 -4.9832e+04  0.0000e+00  0.0000e+00  0.0000e+00]
 [ 1.0091e+08  2.8301e+08  0.0000e+00 -4.9832e+04 -1.9978e+04  0.0000e+00  0.0000e+00  0.0000e+00]
 [ 0.0000e+00  0.0000e+00  8.2139e+07  0.0000e+00  0.0000e+00 -4.9832e+04  0.0000e+00  0.0000e+00]
 [ 1.1964e+05 -4.9832e+04  0.0000e+00  1.2346e+04  2.7507e+03  0.0000e+00  0.0000e+00  0.0000e+00]
 [-4.9832e+04 -1.9978e+04  0.0000e+00  2.7507e+03  6.2773e+03  0.0000e+00  0.0000e+00  0.0000e+00]
 [ 0.0000e+00  0.0000e+00 -4.9832e+04  0.0000e+00  0.0000e+00  2.3503e+03  0.0000e+00  0.0000e+00]
 [ 0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  1.0241e+07 -3.0079e-08]
 [ 0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00 -3.0079e-08  1.1719e+07]]


## All wall laminates: MSG G vs Whitney, and the $U^*$ residual

The complementary-energy (Whitney) shear stiffness is shown for comparison — on sandwich
walls (soft foam core) the MSG least-squares $G$ comes out substantially **softer** than
Whitney, because the second-order energy feels the full core shear compliance.

In [4]:
print("%-9s %2s %8s | %11s %11s | %11s %11s | %9s" %
      ("laminate", "np", "h [m]", "G11_msg", "G22_msg", "G11_Whit", "G22_Whit", "Ustar_rel"))
for ln, lay in layup_db.items():
    thk = [float(t) for t in lay["thick"]]; ang = [float(a) for a in lay["angles"]]
    mats = [str(m) for m in lay["mat_names"]]
    h = float(sum(thk))
    r = rm_plate_msg(thk, ang, mats, mdb, fraction=0.5)
    Gw = transverse_shear_stiffness(thk, ang, mats, mdb)[0]
    G = r["G_msg"]
    print("%-9s %2d %8.4f | %11.4e %11.4e | %11.4e %11.4e | %9.2e" %
          (ln, len(thk), h, G[0, 0], G[1, 1], Gw[0, 0], Gw[1, 1], r["Ustar_rel"]))

laminate  np    h [m] |     G11_msg     G22_msg |    G11_Whit    G22_Whit | Ustar_rel
layup_0    3   0.0160 |  1.0241e+07  1.1719e+07 |  6.9485e+07  6.8360e+07 |  1.33e-02
layup_1    3   0.0520 |  2.8346e+06  2.8366e+06 |  7.4288e+06  7.4284e+06 |  1.39e-04
layup_2    3   0.0460 |  1.1552e+08  2.2027e+07 |  3.6037e+08  2.2073e+08 |  7.65e-04
layup_3    3   0.0760 |  4.1328e+06  4.1366e+06 |  1.1262e+07  1.1262e+07 |  3.03e-04
layup_4    3   0.0110 |  7.4423e+06  8.5658e+06 |  3.8107e+07  3.7956e+07 |  2.50e-02


layup_5    3   0.0400 |  2.1908e+06  2.1926e+06 |  5.5191e+06  5.5187e+06 |  7.97e-05


layup_6    3   0.0410 |  8.2028e+07  1.6388e+07 |  3.1282e+08  1.9303e+08 |  8.88e-04
layup_7    3   0.0730 |  3.9700e+06  3.9735e+06 |  1.0782e+07  1.0782e+07 |  2.79e-04


layup_8    3   0.0420 |  2.2833e+06  2.2833e+06 |  6.1164e+06  6.1164e+06 |  3.64e-03
layup_9    3   0.0280 |  1.5280e+06  1.5280e+06 |  3.8843e+06  3.8843e+06 |  1.42e-03


## Reading the results

* **`Ustar_rel` ≤ ~1e-2 everywhere** — the least-squares projection absorbs almost all of the
  second-order gradient energy into $\gamma^T G \gamma$; the RM wall law is close to
  asymptotically correct for these laminates.  Where it grows (thick soft-core walls), *no*
  choice of $G$ makes the RM form adequate — that is a model limit, not a fitting problem.
* **MSG $G$ vs Whitney** — for the foam-sandwich webs the MSG value is 2–3× softer.  The two
  answer different questions: Whitney is the complementary-energy shear flow of the laminate
  alone; the MSG $G$ is the value that makes the RM *plate model* reproduce the second-order
  asymptotic energy of the 3-D laminate.
* The `G_msg = None` gate: `rm_plate_msg` returns `None` when the fitted compliance
  $X=G^{-1}$ is not positive definite (degenerate placeholder materials can trigger this,
  e.g. the 10-Pa gelcoat of the bundled MH-104 YAML) — fall back to Whitney in that case.
* Theory: Yu 2003 Sec. 4 (Eqs. 55–60) and the equation crosswalk to the IJSS/CMAME twins in
  `docs/MITC_transverse_shear.md`; the 8×8 storage convention matches
  `examples/data/benchmark/st15_rm_plate_8x8.dat`.